<a href="https://colab.research.google.com/github/jcosign/VibeCodingSparta/blob/main/report/week4/%EC%A0%95%EC%97%B0%EC%84%9C_week_04_applied_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week04 Applied Project — 작은 EDA 패키지 만들기

이 노트북은 DataFrame 구조 확인, 결측 처리, 집계, 시각화, 해석 문장 작성을 한 번에 묶는 선택 확장입니다.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 출력 파일이 필요한 셀에서 사용할 폴더입니다.
# Colab에서도 이 폴더가 자동으로 만들어지므로 별도 파일을 받을 필요가 없습니다.
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. 분석용 표 준비

In [2]:
df = pd.DataFrame({
    'date': ['2026-06-01', '2026-06-02', '2026-06-03', '2026-06-04', '2026-06-05', '2026-06-06'],
    'category': ['office', 'life', 'office', 'digital', 'life', 'digital'],
    'sales': [120, 150, None, 110, 90, 180],
    'visits': [300, 360, 280, 310, None, 430],
})
print(df)

         date category  sales  visits
0  2026-06-01   office  120.0   300.0
1  2026-06-02     life  150.0   360.0
2  2026-06-03   office    NaN   280.0
3  2026-06-04  digital  110.0   310.0
4  2026-06-05     life   90.0     NaN
5  2026-06-06  digital  180.0   430.0


## 2. 구조와 결측 확인

In [3]:
print('shape:', df.shape)
print('columns:', list(df.columns))
print('missing:', df.isna().sum().to_dict())

shape: (6, 4)
columns: ['date', 'category', 'sales', 'visits']
missing: {'date': 0, 'category': 0, 'sales': 1, 'visits': 1}


## 3. 사본에서 결측 처리

In [4]:
clean = df.copy()
clean['sales'] = clean['sales'].fillna(clean['sales'].median())
clean['visits'] = clean['visits'].fillna(clean['visits'].median())
print(clean)

         date category  sales  visits
0  2026-06-01   office  120.0   300.0
1  2026-06-02     life  150.0   360.0
2  2026-06-03   office  120.0   280.0
3  2026-06-04  digital  110.0   310.0
4  2026-06-05     life   90.0   310.0
5  2026-06-06  digital  180.0   430.0


## 4. 범주별 요약표 만들기

In [5]:
summary = clean.groupby('category').agg(
    sales_total=('sales', 'sum'),
    sales_mean=('sales', 'mean'),
    visits_total=('visits', 'sum'),
    row_count=('date', 'count'),
).reset_index()
summary['sales_total'] = summary['sales_total'].round(1)
summary['sales_mean'] = summary['sales_mean'].round(1)
print(summary)

  category  sales_total  sales_mean  visits_total  row_count
0  digital        290.0       145.0         740.0          2
1     life        240.0       120.0         670.0          2
2   office        240.0       120.0         580.0          2


## 5. 요약표 저장

In [6]:
summary_path = OUTPUT_DIR / 'week04_applied_summary.csv'
summary.to_csv(summary_path, index=False)
print('saved:', summary_path)

saved: outputs/week04_applied_summary.csv


## 6. 그래프 저장

In [7]:
ax = summary.sort_values('sales_total', ascending=False).plot(
    kind='bar', x='category', y='sales_total', legend=False, title='Sales total by category'
)
ax.set_xlabel('category')
ax.set_ylabel('sales_total')
plt.tight_layout()
chart_path = OUTPUT_DIR / 'week04_applied_sales_total.png'
plt.savefig(chart_path)
plt.close()
print('saved:', chart_path)

saved: outputs/week04_applied_sales_total.png


## 7. 해석 문장 작성

In [8]:
top = summary.sort_values('sales_total', ascending=False).iloc[0]
report = {
    'fact': f"sales_total이 가장 큰 범주는 {top['category']}입니다.",
    'number': float(top['sales_total']),
    'limit': '표본 수가 작으므로 원인을 단정하지 않습니다.',
    'next_check': '기간을 늘리고 캠페인 여부를 함께 확인합니다.',
}
print(report)

{'fact': 'sales_total이 가장 큰 범주는 digital입니다.', 'number': 290.0, 'limit': '표본 수가 작으므로 원인을 단정하지 않습니다.', 'next_check': '기간을 늘리고 캠페인 여부를 함께 확인합니다.'}


## 8. 직접 바꿔보기
아래 기준값을 바꾸면 high_sales 행 수가 달라집니다.

In [9]:
threshold = 130
high_sales = clean[clean['sales'] >= threshold]
print('threshold:', threshold)
print(high_sales[['date', 'category', 'sales']])
print('PROJECT_APPLIED_DONE')

threshold: 130
         date category  sales
1  2026-06-02     life  150.0
5  2026-06-06  digital  180.0
PROJECT_APPLIED_DONE
